# Imports

In [19]:
import re
import cda2
import math

import time
from datetime import datetime, timedelta

import pandas as pd
import pyarrow
from typing import Iterator, Tuple
from pyspark.sql.functions import pandas_udf

from pyspark.sql.window import Window
import pyspark.sql.types as T
import pyspark.sql.functions as F
from pyspark.sql.functions import col, row_number
from pyspark.sql.functions import explode, map_keys, col

In [20]:
api = cda2.Api()

In [21]:
# Set configuration parameters to better optimize queries.

config = {
    "spark.sql.adaptive.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.parallelismFirst": "false",
    "spark.sql.adaptive.coalescePartitions.minPartitionSize": "1m",
    "spark.executor.memory": "8g",
    "spark.executor.memoryOverhead": "16g",
}

In [22]:
# Start Spark and specify number of cpus to use.

api.start_spark(n_executors=100, config=config)

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [5]:
year0 = "2025"
year1 = str(int(year0) + 1)

In [6]:
dates = {"start_date": year0 + "-01-01", "end_date": year1 +"-01-01"}

# retrieve firs

In [7]:
df_firs = (
    api.dataframe("FlightInformationRegion", **dates, metadata=True)
    .select(
        "fir_name",
        "icao_code",
        "region",
        "type",
        F.col("geometry.modules"),
        F.col("metadata.effective_end_date").alias("end_date"),
    )
#    .withColumn("grouping", F.concat("fir_name", F.lit("_"), "region", F.lit("_"), "type", F.lit("_"), "seq"))
    .withColumn("grouping", F.concat("fir_name", F.lit("_"), "region", F.lit("_"), "type"))
    .orderBy("grouping")
)

In [8]:
df_firs.count()

395

## keep only the newest update

In [9]:
window = Window.partitionBy("grouping").orderBy(col("end_date").desc())

df_firs_newest = (df_firs
    .withColumn("row", row_number().over(window))
    .filter(col("row") == 1)
    .drop("row", "end_date")
    .orderBy("grouping")
)

In [10]:
#df_firs_newest.count()

In [11]:
#df_firs_newest.show(10)

In [12]:
modules_schema = T.StructType([
    T.StructField("modules", T.ArrayType(
        T.StructType([
            T.StructField("identifier", T.DoubleType(), True),
            T.StructField("floor", T.DoubleType(), True),
            T.StructField("ceiling", T.DoubleType(), True),
            T.StructField("polygon", T.StructType([
                T.StructField("boundary", T.ArrayType(
                    T.StructType([
                        T.StructField("latitude", T.DoubleType(), True),
                        T.StructField("longitude", T.DoubleType(), True),
                        T.StructField("sequence_number", T.IntegerType(), True),
                    ]), True), True),
            ]), True),
        ]), True), True),
    ])

In [13]:
none_string = "*"
separator_string = ":"

#@F.udf(T.StringType())
@F.udf(T.MapType(T.StringType(),T.StringType()))
def extract_boundary(array_of_modules:T.ArrayType(modules_schema)) -> map:  
    result = {}
    
    for module in array_of_modules:
        result["floor"] = f'{module.floor:.0f}'
        result["ceiling"] = f'{module.ceiling:.0f}'

        boundary_string = ""

        module.polygon.boundary.sort(key=lambda x: x.sequence_number, reverse=False)

        for point in module.polygon.boundary:
            boundary_string += (none_string if point.latitude is None else f'{point.latitude:.6f}') + " "
            boundary_string += (none_string if point.longitude is None else f'{point.longitude:.6f}') + separator_string

        # drop the trailing separator
        if len(boundary_string) > 0:
            boundary_string = boundary_string[0:len(separator_string) * -1]
           
        result["boundary"] = boundary_string
        
    return result

In [14]:
df_firs_dict = (
    df_firs_newest
    .withColumn("module_dict", extract_boundary("modules"))
    .select(
        "type",
        "region",
        "icao_code",
        "fir_name",
        "module_dict",
    )
    .drop("modules")
    .orderBy("type","region","icao_code","fir_name")
)

In [15]:
# expand the dictionary to columns

# https://mungingdata.com/pyspark/dict-map-to-multiple-columns/
# https://stackoverflow.com/questions/36869134/pyspark-converting-a-column-of-type-map-to-multiple-columns-in-a-dataframe

keys = ["ceiling", "floor", "boundary"]
key_cols = list(map(lambda f: F.col("module_dict").getItem(f).alias(str(f)), keys))
final_cols = [
        "type",
        "region",
        "icao_code",
        "fir_name",
             ] + key_cols

In [16]:
df_firs_output = (
    df_firs_dict.select(final_cols)
)

In [17]:
#df_firs_output.show(10)

In [18]:
(
    df_firs_output
    .write.option("header", True)
    .csv("CRAFT/" + year0 + "/firs", compression="None", mode="overwrite")
)